In [1]:
# ============================================================
# CELL 1: Imports & Configuration
# ============================================================

import sqlite3
import pandas as pd
import numpy as np
import warnings
import time
from datetime import datetime

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('future.no_silent_downcasting', True)

DB_PATH = r'C:\Users\Sarthak\Documents\ML\fighter-beta\mma_fighters.db'
print("Imports complete")

C:\Users\Sarthak\miniconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Imports complete


In [2]:
# ============================================================
# CELL 2: Database Connection & Verification
# ============================================================

conn = sqlite3.connect(DB_PATH)

# Verify tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"Connected. Tables: {tables['name'].tolist()}")

# Quick data check
fights = pd.read_sql("SELECT COUNT(*) as count FROM fights_v2", conn)
fighters = pd.read_sql("SELECT COUNT(*) as count FROM fighters_v2", conn)
rounds = pd.read_sql("SELECT COUNT(*) as count FROM fight_round_stats_v2", conn)

print(f"Fights: {fights['count'].iloc[0]}")
print(f"Fighters: {fighters['count'].iloc[0]}")
print(f"Rounds: {rounds['count'].iloc[0]}")

Connected. Tables: ['fighters_v2', 'fights_v2', 'fight_fighters_v2', 'fight_rounds_v2', 'fight_round_stats_v2', 'fight_round_sig_strikes_v2', 'fight_totals_v2', 'ingestion_fighter_progress_v2', 'fight_odds', 'odds_scrape_progress']
Fights: 11127
Fighters: 4451
Rounds: 48772


In [3]:
# ============================================================
# CELL 3: Helper Functions
# ============================================================

def parse_reach(r):
    if pd.isna(r) or r == '--': return None
    try:
        return float(r.replace('"', ''))
    except:
        return None

def parse_height(h):
    if pd.isna(h) or h == '--': return None
    try:
        parts = h.replace('"', '').split("'")
        return int(parts[0]) * 12 + int(parts[1])
    except:
        return None

def parse_age(dob, fight_date):
    if pd.isna(dob) or pd.isna(fight_date): return None
    try:
        birth = datetime.strptime(dob, "%b %d, %Y")
        fight = datetime.strptime(fight_date, "%Y-%m-%d")
        return (fight - birth).days / 365.25
    except:
        return None

def parse_fight_duration(ending_round, ending_time):
    """Calculate actual fight duration in minutes"""
    try:
        mins, secs = ending_time.split(':')
        final_round_minutes = int(mins) + int(secs) / 60
        return ((ending_round - 1) * 5) + final_round_minutes
    except:
        return 15.0  # Default 3 round fight

def time_decay_weights(dates, as_of_date, lam=0.005):
    """
    Calculate exponential time decay weights.
    lam=0.005 = ~1.3 year half-life (moderate decay)
    
    More recent fights get higher weights.
    """
    as_of = datetime.strptime(as_of_date, "%Y-%m-%d")
    weights = []
    for d in dates:
        fight_dt = datetime.strptime(d, "%Y-%m-%d")
        days_ago = (as_of - fight_dt).days
        w = np.exp(-lam * days_ago)
        weights.append(w)
    weights = np.array(weights)
    # Normalize so weights sum to 1
    return weights / weights.sum() if weights.sum() > 0 else weights

print("Helper functions ready")

Helper functions ready


In [26]:
# ============================================================
# CELL 4 (Updated): Pre-Cache with Strike Breakdown
# ============================================================

print("Fetching all fight stats with strike breakdown...")
start = time.time()

# Original fight totals
all_fight_stats = pd.read_sql("""
    SELECT 
        f.fight_id,
        f.event_date,
        f.weight_class,
        f.ending_round,
        f.ending_time,
        f.method,
        ff.fighter_id,
        SUM(frs.sig_str_landed) as sig_str_landed,
        SUM(frs.sig_str_attempted) as sig_str_attempted,
        SUM(frs.td_landed) as td_landed,
        SUM(frs.td_attempted) as td_attempted,
        SUM(frs.sub_attempts) as sub_attempts,
        SUM(frs.ctrl_seconds) as ctrl_seconds,
        SUM(frs.knockdowns) as knockdowns,
        COUNT(DISTINCT fr.round_number) as total_rounds
    FROM fight_round_stats_v2 frs
    JOIN fight_rounds_v2 fr ON frs.round_id = fr.round_id
    JOIN fights_v2 f ON fr.fight_id = f.fight_id
    JOIN fight_fighters_v2 ff ON f.fight_id = ff.fight_id AND frs.fighter_id = ff.fighter_id
    GROUP BY f.fight_id, ff.fighter_id
""", conn)

# Strike breakdown (head/body/leg, distance/clinch/ground)
strike_breakdown = pd.read_sql("""
    SELECT 
        fr.fight_id,
        frss.fighter_id,
        SUM(frss.head_landed) as head_landed,
        SUM(frss.body_landed) as body_landed,
        SUM(frss.leg_landed) as leg_landed,
        SUM(frss.distance_landed) as distance_landed,
        SUM(frss.clinch_landed) as clinch_landed,
        SUM(frss.ground_landed) as ground_landed
    FROM fight_round_sig_strikes_v2 frss
    JOIN fight_rounds_v2 fr ON frss.round_id = fr.round_id
    GROUP BY fr.fight_id, frss.fighter_id
""", conn)

# Merge strike breakdown into main stats
all_fight_stats = all_fight_stats.merge(
    strike_breakdown, 
    on=['fight_id', 'fighter_id'], 
    how='left'
).fillna(0)

# Calculate actual fight duration
all_fight_stats['actual_minutes'] = all_fight_stats.apply(
    lambda r: parse_fight_duration(r['ending_round'], r['ending_time']), axis=1
)

# Calculate per-minute stats (original)
all_fight_stats['slpm']             = all_fight_stats['sig_str_landed'] / all_fight_stats['actual_minutes']
all_fight_stats['str_acc']          = all_fight_stats['sig_str_landed'] / all_fight_stats['sig_str_attempted']
all_fight_stats['td_acc']           = all_fight_stats['td_landed'] / all_fight_stats['td_attempted']
all_fight_stats['td_avg']           = all_fight_stats['td_landed'] / (all_fight_stats['actual_minutes'] / 15)
all_fight_stats['sub_avg']          = all_fight_stats['sub_attempts'] / (all_fight_stats['actual_minutes'] / 15)
all_fight_stats['ctrl_time_per_min']= all_fight_stats['ctrl_seconds'] / all_fight_stats['actual_minutes'] / 60
all_fight_stats['kd_per_min']       = all_fight_stats['knockdowns'] / all_fight_stats['actual_minutes']

# Calculate per-minute stats (strike breakdown)
all_fight_stats['head_slpm']     = all_fight_stats['head_landed'] / all_fight_stats['actual_minutes']
all_fight_stats['body_slpm']     = all_fight_stats['body_landed'] / all_fight_stats['actual_minutes']
all_fight_stats['leg_slpm']      = all_fight_stats['leg_landed'] / all_fight_stats['actual_minutes']
all_fight_stats['distance_slpm'] = all_fight_stats['distance_landed'] / all_fight_stats['actual_minutes']
all_fight_stats['clinch_slpm']   = all_fight_stats['clinch_landed'] / all_fight_stats['actual_minutes']
all_fight_stats['ground_slpm']   = all_fight_stats['ground_landed'] / all_fight_stats['actual_minutes']

all_fight_stats = all_fight_stats.replace([np.inf, -np.inf], np.nan).fillna(0)
all_fight_stats = all_fight_stats.sort_values('event_date').reset_index(drop=True)

# Opponent lookup
all_opponents = pd.read_sql("""
    SELECT ff1.fight_id, ff1.fighter_id, ff2.fighter_id as opponent_id
    FROM fight_fighters_v2 ff1
    JOIN fight_fighters_v2 ff2 
        ON ff1.fight_id = ff2.fight_id 
        AND ff1.fighter_id != ff2.fighter_id
""", conn)

# Build O(1) lookup dicts
fighter_fights_dict = {
    fid: group.sort_values('event_date').reset_index(drop=True)
    for fid, group in all_fight_stats.groupby('fighter_id')
}

opponents_dict = {
    (row['fight_id'], row['fighter_id']): row['opponent_id']
    for _, row in all_opponents.iterrows()
}

print(f"Done in {time.time()-start:.1f}s")
print(f"Fighters cached: {len(fighter_fights_dict)}")
print(f"Opponent lookups: {len(opponents_dict)}")
print(f"New stats available: head_slpm, body_slpm, leg_slpm, distance_slpm, clinch_slpm, ground_slpm")

Fetching all fight stats with strike breakdown...
Done in 4.3s
Fighters cached: 3760
Opponent lookups: 22254
New stats available: head_slpm, body_slpm, leg_slpm, distance_slpm, clinch_slpm, ground_slpm


In [5]:
# ============================================================
# CELL 5: Generate Base Fight Dataset
# ============================================================

def generate_base_fights(start_date='2007-01-01', end_date='2026-12-31', conn=conn):
    """
    Generate one row per fight with metadata.
    This is the foundation - all features are joined to this.
    """
    fights = pd.read_sql(f"""
        SELECT 
            f.fight_id,
            f.event_date,
            f.event_name,
            f.weight_class,
            f.method,
            f.ending_round,
            f.ending_time,
            ff1.fighter_id as fighter_1_id,
            fv1.name as fighter_1_name,
            ff2.fighter_id as fighter_2_id,
            fv2.name as fighter_2_name,
            CASE WHEN ff1.result = 'win' THEN 1 ELSE 0 END as winner
        FROM fights_v2 f
        JOIN fight_fighters_v2 ff1 ON f.fight_id = ff1.fight_id AND ff1.corner = 'fighter_1'
        JOIN fight_fighters_v2 ff2 ON f.fight_id = ff2.fight_id AND ff2.corner = 'fighter_2'
        JOIN fighters_v2 fv1 ON ff1.fighter_id = fv1.fighter_id
        JOIN fighters_v2 fv2 ON ff2.fighter_id = fv2.fighter_id
        WHERE f.event_date >= '{start_date}'
        AND f.event_date <= '{end_date}'
        ORDER BY f.event_date
    """, conn)
    
    return fights

# Generate base dataset
base_fights = generate_base_fights(start_date='2007-01-01', end_date='2022-12-31')
print(f"Base fights shape: {base_fights.shape}")
print(f"Date range: {base_fights['event_date'].min()} to {base_fights['event_date'].max()}")
print(f"Winner distribution: {base_fights['winner'].value_counts().to_dict()}")

Base fights shape: (7800, 12)
Date range: 2007-01-20 to 2022-12-17
Winner distribution: {0: 3918, 1: 3882}


In [6]:
# ============================================================
# CELL 6: Decayed Rolling Stats
# ============================================================

def calculate_decayed_rolling_stats(fighter_id, as_of_date, window=5, lam=0.005):
    """
    Calculate fighter's rolling stats with TIME DECAY.
    Recent fights weighted more than older fights.
    
    Args:
        fighter_id: Fighter's ID
        as_of_date: Only use fights before this date
        window: Last N fights to use
        lam: Decay rate (0.005 = ~1.3 year half-life)
    
    Returns:
        dict of weighted stats, or None if no history
    """
    
    if fighter_id not in fighter_fights_dict:
        return None
    
    fights = fighter_fights_dict[fighter_id]
    recent = fights[fights['event_date'] < as_of_date].tail(window)
    
    if len(recent) == 0:
        return None
    
    # Calculate time decay weights
    weights = time_decay_weights(recent['event_date'].tolist(), as_of_date, lam)
    
    stats = {}
    stat_cols = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg', 'ctrl_time_per_min', 'kd_per_min']
    
    for stat in stat_cols:
        values = recent[stat].values
        # Weighted average
        stats[stat] = np.average(values, weights=weights)
    
    stats['n_fights'] = len(recent)
    
    return stats


# Test it
sample_fight = base_fights.iloc[100]
f1_stats = calculate_decayed_rolling_stats(
    sample_fight['fighter_1_id'], 
    sample_fight['event_date'],
    window=5
)

print(f"Fighter: {sample_fight['fighter_1_name']}")
print(f"Fight date: {sample_fight['event_date']}")
print(f"Decayed rolling stats (last 5):")
print(f1_stats)

Fighter: Alessio Sakara
Fight date: 2007-04-21
Decayed rolling stats (last 5):
{'slpm': 5.727086679596848, 'str_acc': 0.4928067907876295, 'td_acc': 0.13334881699470413, 'td_avg': 1.2001393529523372, 'sub_avg': 0.0, 'ctrl_time_per_min': 0.37859097854623225, 'kd_per_min': 0.014164525630851577, 'n_fights': 4}


In [27]:
# ============================================================
# CELL 7 (Updated): Decayed Rolling Stats with Strike Breakdown
# ============================================================

import math
LAM = math.log(2) / (1.5 * 365)
print(f"Using lambda: {LAM:.6f}")

def calculate_decayed_rolling_stats(fighter_id, as_of_date, window=5, lam=LAM):
    
    if fighter_id not in fighter_fights_dict:
        return None
    
    fights = fighter_fights_dict[fighter_id]
    recent = fights[fights['event_date'] < as_of_date].tail(window)
    
    if len(recent) == 0:
        return None
    
    weights = time_decay_weights(recent['event_date'].tolist(), as_of_date, lam)
    
    stat_cols = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg', 'ctrl_time_per_min', 'kd_per_min',
                 'head_slpm', 'body_slpm', 'leg_slpm', 'distance_slpm', 'clinch_slpm', 'ground_slpm']
    
    stats = {}
    for stat in stat_cols:
        stats[stat] = np.average(recent[stat].values, weights=weights)
    
    stats['n_fights'] = len(recent)
    return stats


def generate_decayed_features(base_df, window=5, lam=LAM):
    
    stat_cols = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg', 'ctrl_time_per_min', 'kd_per_min',
                 'head_slpm', 'body_slpm', 'leg_slpm', 'distance_slpm', 'clinch_slpm', 'ground_slpm']
    
    rows = []
    
    for idx, fight in base_df.iterrows():
        if idx % 1000 == 0:
            print(f"  Processing {idx}/{len(base_df)}...")
        
        row = {'fight_id': fight['fight_id']}
        
        for prefix, fid in [('f1', fight['fighter_1_id']), ('f2', fight['fighter_2_id'])]:
            stats = calculate_decayed_rolling_stats(fid, fight['event_date'], window, lam)
            
            for stat in stat_cols:
                row[f'{prefix}_{stat}'] = stats[stat] if stats else 0
            
            # Experience features
            if fid in fighter_fights_dict:
                history = fighter_fights_dict[fid]
                past = history[history['event_date'] < fight['event_date']]
                row[f'{prefix}_total_fights'] = len(past)
                
                if len(past) > 0:
                    last_date = datetime.strptime(past.iloc[-1]['event_date'], "%Y-%m-%d")
                    fight_date = datetime.strptime(fight['event_date'], "%Y-%m-%d")
                    row[f'{prefix}_days_since_last'] = (fight_date - last_date).days
                else:
                    row[f'{prefix}_days_since_last'] = 0
            else:
                row[f'{prefix}_total_fights'] = 0
                row[f'{prefix}_days_since_last'] = 0
        
        # Difference features only
        for stat in stat_cols:
            row[f'diff_{stat}'] = row[f'f1_{stat}'] - row[f'f2_{stat}']
        
        row['diff_total_fights']    = row['f1_total_fights'] - row['f2_total_fights']
        row['diff_days_since_last'] = row['f1_days_since_last'] - row['f2_days_since_last']
        
        rows.append(row)
    
    return base_df.merge(pd.DataFrame(rows), on='fight_id', how='left')


# Regenerate with strike breakdown
print("Generating features with strike breakdown...")
start = time.time()
fight_features = generate_decayed_features(base_fights, window=5, lam=LAM)
fight_features = add_physical_features(fight_features, conn)
print(f"Done in {time.time()-start:.1f}s")
print(f"Shape: {fight_features.shape}")

Using lambda: 0.001266
Generating features with strike breakdown...
  Processing 0/7800...
  Processing 1000/7800...
  Processing 2000/7800...
  Processing 3000/7800...
  Processing 4000/7800...
  Processing 5000/7800...
  Processing 6000/7800...
  Processing 7000/7800...
Done in 19.1s
Shape: (7800, 70)


In [8]:
# Find a fighter with many fights close together
active_fighter = pd.read_sql("""
    SELECT ff.fighter_id, fv.name, COUNT(*) as fight_count
    FROM fight_fighters_v2 ff
    JOIN fighters_v2 fv ON ff.fighter_id = fv.fighter_id
    JOIN fights_v2 f ON ff.fight_id = f.fight_id
    WHERE f.event_date BETWEEN '2018-01-01' AND '2022-01-01'
    GROUP BY ff.fighter_id
    ORDER BY fight_count DESC
    LIMIT 5
""", conn)
print(active_fighter)

         fighter_id             name  fight_count
0  3a46b268013afede    Kevin Holland           14
1  f0feeb2192937424      Angela Hill           13
2  3738e68d2261e60f  Andrei Arlovski           12
3  d4c9dcd330403612    Anthony Smith           11
4  d3df1add9d9a7efb    Derrick Lewis           11


In [9]:
# Test decay on Kevin Holland (14 fights, very active)
holland_id = active_fighter['fighter_id'].iloc[0]
test_date = '2022-01-01'

raw_fights = fighter_fights_dict[holland_id]
raw_fights = raw_fights[raw_fights['event_date'] < test_date].tail(5)
print("Kevin Holland last 5 fights (raw slpm):")
print(raw_fights[['event_date', 'slpm']])

weights = time_decay_weights(raw_fights['event_date'].tolist(), test_date, lam=0.005)
print(f"\nDecay weights: {weights.round(3)}")

weighted_slpm = np.average(raw_fights['slpm'].values, weights=weights)
simple_avg = raw_fights['slpm'].mean()

print(f"\nSimple average slpm: {simple_avg:.2f}")
print(f"Decayed average slpm: {weighted_slpm:.2f}")
print(f"Difference: {weighted_slpm - simple_avg:.2f}")

Kevin Holland last 5 fights (raw slpm):
    event_date      slpm
9   2020-10-31  4.150943
10  2020-12-12  8.571429
11  2021-03-20  1.440000
12  2021-04-10  1.480000
13  2021-10-02  0.807175

Decay weights: [0.084 0.104 0.17  0.189 0.453]

Simple average slpm: 3.29
Decayed average slpm: 2.13
Difference: -1.16


In [41]:
# ============================================================
# CELL 7 (Updated): Add Accuracy Stats to Rolling Features
# ============================================================

import math
LAM = math.log(2) / (1.5 * 365)
print(f"Using lambda: {LAM:.6f}")

def calculate_decayed_rolling_stats(fighter_id, as_of_date, window=5, lam=LAM):
    
    if fighter_id not in fighter_fights_dict:
        return None
    
    fights = fighter_fights_dict[fighter_id]
    recent = fights[fights['event_date'] < as_of_date].tail(window)
    
    if len(recent) == 0:
        return None
    
    weights = time_decay_weights(recent['event_date'].tolist(), as_of_date, lam)
    
    stat_cols = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg', 'ctrl_time_per_min', 'kd_per_min',
                 'head_slpm', 'body_slpm', 'leg_slpm', 'distance_slpm', 'clinch_slpm', 'ground_slpm',
                 'head_acc', 'body_acc', 'leg_acc', 'distance_acc', 'clinch_acc', 'ground_acc']
    
    stats = {}
    for stat in stat_cols:
        stats[stat] = np.average(recent[stat].values, weights=weights)
    
    stats['n_fights'] = len(recent)
    return stats


def generate_decayed_features(base_df, window=5, lam=LAM):
    
    stat_cols = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg', 'ctrl_time_per_min', 'kd_per_min',
                 'head_slpm', 'body_slpm', 'leg_slpm', 'distance_slpm', 'clinch_slpm', 'ground_slpm',
                 'head_acc', 'body_acc', 'leg_acc', 'distance_acc', 'clinch_acc', 'ground_acc']
    
    rows = []
    
    for idx, fight in base_df.iterrows():
        if idx % 1000 == 0:
            print(f"  Processing {idx}/{len(base_df)}...")
        
        row = {'fight_id': fight['fight_id']}
        
        for prefix, fid in [('f1', fight['fighter_1_id']), ('f2', fight['fighter_2_id'])]:
            stats = calculate_decayed_rolling_stats(fid, fight['event_date'], window, lam)
            
            for stat in stat_cols:
                row[f'{prefix}_{stat}'] = stats[stat] if stats else 0
            
            if fid in fighter_fights_dict:
                history = fighter_fights_dict[fid]
                past = history[history['event_date'] < fight['event_date']]
                row[f'{prefix}_total_fights'] = len(past)
                
                if len(past) > 0:
                    last_date = datetime.strptime(past.iloc[-1]['event_date'], "%Y-%m-%d")
                    fight_date = datetime.strptime(fight['event_date'], "%Y-%m-%d")
                    row[f'{prefix}_days_since_last'] = (fight_date - last_date).days
                else:
                    row[f'{prefix}_days_since_last'] = 0
            else:
                row[f'{prefix}_total_fights'] = 0
                row[f'{prefix}_days_since_last'] = 0
        
        for stat in stat_cols:
            row[f'diff_{stat}'] = row[f'f1_{stat}'] - row[f'f2_{stat}']
        
        row['diff_total_fights']    = row['f1_total_fights'] - row['f2_total_fights']
        row['diff_days_since_last'] = row['f1_days_since_last'] - row['f2_days_since_last']
        
        rows.append(row)
    
    return base_df.merge(pd.DataFrame(rows), on='fight_id', how='left')


print("Regenerating with accuracy stats...")
start = time.time()
fight_features = generate_decayed_features(base_fights, window=5, lam=LAM)
fight_features = add_physical_features(fight_features, conn)
print(f"Done in {time.time()-start:.1f}s")
print(f"Shape: {fight_features.shape}")

Using lambda: 0.001266
Regenerating with accuracy stats...
  Processing 0/7800...
  Processing 1000/7800...
  Processing 2000/7800...
  Processing 3000/7800...
  Processing 4000/7800...
  Processing 5000/7800...
  Processing 6000/7800...
  Processing 7000/7800...
Done in 22.2s
Shape: (7800, 88)


In [18]:
# ============================================================
# CELL 8: Physical Features
# ============================================================

def add_physical_features(df, conn):
    
    all_ids = set(df['fighter_1_id'].unique()) | set(df['fighter_2_id'].unique())
    
    fighters = pd.read_sql(f"""
        SELECT fighter_id, reach, height, dob, stance
        FROM fighters_v2
        WHERE fighter_id IN ({','.join(f"'{fid}'" for fid in all_ids)})
    """, conn)
    
    fighters['reach_in'] = fighters['reach'].apply(parse_reach)
    fighters['height_in'] = fighters['height'].apply(parse_height)
    
    # Merge fighter 1
    df = df.merge(
        fighters[['fighter_id', 'reach_in', 'height_in', 'dob', 'stance']],
        left_on='fighter_1_id', right_on='fighter_id', how='left'
    ).drop('fighter_id', axis=1).rename(columns={
        'reach_in': 'f1_reach', 'height_in': 'f1_height', 
        'dob': 'f1_dob', 'stance': 'f1_stance'
    })
    
    # Merge fighter 2
    df = df.merge(
        fighters[['fighter_id', 'reach_in', 'height_in', 'dob', 'stance']],
        left_on='fighter_2_id', right_on='fighter_id', how='left'
    ).drop('fighter_id', axis=1).rename(columns={
        'reach_in': 'f2_reach', 'height_in': 'f2_height',
        'dob': 'f2_dob', 'stance': 'f2_stance'
    })
    
    # Calculate age at fight date
    df['f1_age'] = df.apply(lambda r: parse_age(r['f1_dob'], r['event_date']), axis=1)
    df['f2_age'] = df.apply(lambda r: parse_age(r['f2_dob'], r['event_date']), axis=1)
    
    # Differences
    df['reach_diff']  = df['f1_reach'] - df['f2_reach']
    df['height_diff'] = df['f1_height'] - df['f2_height']
    df['age_diff']    = df['f1_age'] - df['f2_age']
    
    # Ratios
    df['reach_ratio']  = df['f1_reach'] / df['f2_reach']
    df['height_ratio'] = df['f1_height'] / df['f2_height']
    df['age_ratio']    = df['f1_age'] / df['f2_age']
    
    # Stance matchup
    df['stance_matchup'] = (df['f1_stance'] != df['f2_stance']).astype(int)
    
    # Drop raw cols not needed for training
    df = df.drop(['f1_dob', 'f2_dob', 'f1_stance', 'f2_stance'], axis=1)
    
    return df

# Run
fight_features = add_physical_features(fight_features, conn)
print(f"Shape: {fight_features.shape}")
print(f"New cols: {['reach_diff','height_diff','age_diff','reach_ratio','height_ratio','age_ratio','stance_matchup']}")

ValueError: Columns must be same length as key

In [19]:
print(fight_features.columns.tolist())
print(f"Shape: {fight_features.shape}")

['fight_id', 'event_date', 'event_name', 'weight_class', 'method', 'ending_round', 'ending_time', 'fighter_1_id', 'fighter_1_name', 'fighter_2_id', 'fighter_2_name', 'winner', 'f1_slpm', 'f1_str_acc', 'f1_td_acc', 'f1_td_avg', 'f1_sub_avg', 'f1_ctrl_time_per_min', 'f1_kd_per_min', 'f1_total_fights', 'f1_days_since_last', 'f2_slpm', 'f2_str_acc', 'f2_td_acc', 'f2_td_avg', 'f2_sub_avg', 'f2_ctrl_time_per_min', 'f2_kd_per_min', 'f2_total_fights', 'f2_days_since_last', 'diff_slpm', 'diff_str_acc', 'diff_td_acc', 'diff_td_avg', 'diff_sub_avg', 'diff_ctrl_time_per_min', 'diff_kd_per_min', 'diff_total_fights', 'diff_days_since_last', 'f1_reach', 'f1_height', 'f2_reach', 'f2_height', 'f1_age', 'f2_age', 'reach_diff', 'height_diff', 'age_diff', 'reach_ratio', 'height_ratio', 'age_ratio', 'stance_matchup']
Shape: (7800, 52)


In [12]:
def generate_decayed_features(base_df, window=5, lam=0.005):
    
    stat_cols = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg', 'ctrl_time_per_min', 'kd_per_min']
    rows = []
    
    for idx, fight in base_df.iterrows():
        if idx % 1000 == 0:
            print(f"  Processing {idx}/{len(base_df)}...")
        
        row = {'fight_id': fight['fight_id']}
        
        for prefix, fid in [('f1', fight['fighter_1_id']), ('f2', fight['fighter_2_id'])]:
            stats = calculate_decayed_rolling_stats(fid, fight['event_date'], window, lam)
            
            for stat in stat_cols:
                row[f'{prefix}_{stat}'] = stats[stat] if stats else 0
            
            # Experience features
            if fid in fighter_fights_dict:
                history = fighter_fights_dict[fid]
                past = history[history['event_date'] < fight['event_date']]
                row[f'{prefix}_total_fights'] = len(past)
                
                if len(past) > 0:
                    last_date = datetime.strptime(past.iloc[-1]['event_date'], "%Y-%m-%d")
                    fight_date = datetime.strptime(fight['event_date'], "%Y-%m-%d")
                    row[f'{prefix}_days_since_last'] = (fight_date - last_date).days
                else:
                    row[f'{prefix}_days_since_last'] = 0
            else:
                row[f'{prefix}_total_fights'] = 0
                row[f'{prefix}_days_since_last'] = 0
        
        # Differences
        for stat in stat_cols:
            row[f'diff_{stat}'] = row[f'f1_{stat}'] - row[f'f2_{stat}']
        
        row['diff_total_fights'] = row['f1_total_fights'] - row['f2_total_fights']
        row['diff_days_since_last'] = row['f1_days_since_last'] - row['f2_days_since_last']
        
        rows.append(row)
    
    return base_df.merge(pd.DataFrame(rows), on='fight_id', how='left')

# Regenerate
print("Regenerating with experience features...")
start = time.time()
fight_features = generate_decayed_features(base_fights, window=5, lam=0.005)
fight_features = add_physical_features(fight_features, conn)
print(f"Done in {time.time()-start:.1f}s")
print(f"Shape: {fight_features.shape}")

Regenerating with experience features...
  Processing 0/7800...
  Processing 1000/7800...
  Processing 2000/7800...
  Processing 3000/7800...
  Processing 4000/7800...
  Processing 5000/7800...
  Processing 6000/7800...
  Processing 7000/7800...
Done in 15.6s
Shape: (7800, 52)


In [20]:
# ============================================================
# CELL 10: Baseline Model Test
# ============================================================

drop_cols = ['fight_id', 'event_date', 'event_name', 'weight_class', 'method',
             'ending_round', 'ending_time', 'fighter_1_id', 'fighter_1_name',
             'fighter_2_id', 'fighter_2_name', 'winner',
             'f1_reach', 'f2_reach', 'f1_height', 'f2_height', 'f1_age', 'f2_age']

train_data = fight_features[fight_features['event_date'] < '2021-01-01']
val_data   = fight_features[fight_features['event_date'] >= '2021-01-01']

X_train = train_data.drop(drop_cols, axis=1).fillna(0)
y_train = train_data['winner']
X_val   = val_data.drop(drop_cols, axis=1).fillna(0)
y_val   = val_data['winner']

print(f"Train: {len(X_train)} fights | Val: {len(X_val)} fights")
print(f"Features: {X_train.shape[1]}")

model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    min_child_weight=8,
    subsample=0.7,
    colsample_bytree=0.7,
    gamma=1.0,
    random_state=42
)

model.fit(X_train, y_train)
print(f"\nTraining:   {model.score(X_train, y_train):.1%}")
print(f"Validation: {model.score(X_val, y_val):.1%}")

Train: 6654 fights | Val: 1146 fights
Features: 34

Training:   70.5%
Validation: 58.1%


In [14]:
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_df.head(20).to_string())

                 feature  importance
32             age_ratio    0.053927
25     diff_total_fights    0.045458
29              age_diff    0.040776
21           diff_td_avg    0.036922
18             diff_slpm    0.036063
19          diff_str_acc    0.034978
30           reach_ratio    0.032938
6          f1_kd_per_min    0.029892
9                f2_slpm    0.029793
27            reach_diff    0.029313
5   f1_ctrl_time_per_min    0.029289
4             f1_sub_avg    0.029116
0                f1_slpm    0.028633
16       f2_total_fights    0.028090
8     f1_days_since_last    0.027785
17    f2_days_since_last    0.027776
10            f2_str_acc    0.027573
7        f1_total_fights    0.027536
24       diff_kd_per_min    0.027197
1             f1_str_acc    0.026950


In [15]:
# Only diff + physical features
keep_cols = [c for c in X_train.columns if c.startswith('diff_') 
             or c in ['age_diff', 'reach_diff', 'height_diff', 
                      'age_ratio', 'reach_ratio', 'height_ratio', 
                      'stance_matchup']]

print(f"Features: {len(keep_cols)}")
print(keep_cols)

model2 = XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    min_child_weight=8, subsample=0.7, colsample_bytree=0.7,
    gamma=1.0, random_state=42
)

model2.fit(X_train[keep_cols], y_train)
print(f"\nTraining:   {model2.score(X_train[keep_cols], y_train):.1%}")
print(f"Validation: {model2.score(X_val[keep_cols], y_val):.1%}")

Features: 16
['diff_slpm', 'diff_str_acc', 'diff_td_acc', 'diff_td_avg', 'diff_sub_avg', 'diff_ctrl_time_per_min', 'diff_kd_per_min', 'diff_total_fights', 'diff_days_since_last', 'reach_diff', 'height_diff', 'age_diff', 'reach_ratio', 'height_ratio', 'age_ratio', 'stance_matchup']

Training:   67.6%
Validation: 57.9%


In [16]:
# Calculate lambda for 1.5 year half-life
import math

half_life_days = 1.5 * 365  # 547.5 days
lam = math.log(2) / half_life_days
print(f"Lambda for 1.5 year half-life: {lam:.6f}")

# Verify
test_weight = math.exp(-lam * half_life_days)
print(f"Weight at 1.5 years ago: {test_weight:.3f} (should be 0.5)")

# Compare to what we used
print(f"\nOur lambda: 0.005")
print(f"Correct lambda: {lam:.6f}")
print(f"Difference: {lam - 0.005:.6f}")

Lambda for 1.5 year half-life: 0.001266
Weight at 1.5 years ago: 0.500 (should be 0.5)

Our lambda: 0.005
Correct lambda: 0.001266
Difference: -0.003734


In [21]:
# ============================================================
# CELL 11: AdjPerf (Opponent-Adjusted Performance)
# ============================================================

def calculate_weight_class_prior(weight_class, stat_name, use_std=True):
    """Calculate weight class baseline (median + std or MAD)"""
    
    wc_fights = all_fight_stats[all_fight_stats['weight_class'].str.contains(weight_class, na=False)]
    
    if len(wc_fights) == 0:
        return None
    
    values = wc_fights[stat_name].values
    median = np.median(values)
    
    if use_std:
        spread = np.std(values)
    else:
        spread = np.median(np.abs(values - median))  # MAD
    
    return {
        'median': median,
        'spread': spread if spread > 0 else 0.01
    }


def calculate_opponent_adjperf_baseline(opponent_id, stat_name, as_of_date, window=10, use_std=True):
    """
    Calculate what this opponent typically ALLOWS (their defensive baseline).
    Uses time-decayed average with 1.5 year half-life.
    """
    
    if opponent_id not in fighter_fights_dict:
        return None
    
    opp_fights = fighter_fights_dict[opponent_id]
    recent = opp_fights[opp_fights['event_date'] < as_of_date].tail(window)
    
    if len(recent) == 0:
        return None
    
    # Time decay weights
    weights = time_decay_weights(recent['event_date'].tolist(), as_of_date, LAM)
    values = recent[stat_name].values
    
    # Weighted stats
    weighted_median = np.average(values, weights=weights)
    
    if use_std:
        # Weighted standard deviation
        variance = np.average((values - weighted_median)**2, weights=weights)
        spread = np.sqrt(variance)
    else:
        # Weighted MAD
        abs_devs = np.abs(values - weighted_median)
        spread = np.average(abs_devs, weights=weights)
    
    return {
        'median': weighted_median,
        'spread': spread if spread > 0 else 0.01,
        'n_fights': len(recent)
    }


def calculate_adjperf_zscore(observed, opponent_baseline, weight_class_prior, shrinkage_k=5):
    """Calculate z-score with Bayesian shrinkage"""
    
    if opponent_baseline is None:
        median = weight_class_prior['median']
        spread = weight_class_prior['spread']
    else:
        # Bayesian shrinkage
        n_eff = opponent_baseline['n_fights']
        weight = n_eff / (n_eff + shrinkage_k)
        
        median = weight * opponent_baseline['median'] + (1 - weight) * weight_class_prior['median']
        spread = weight * opponent_baseline['spread'] + (1 - weight) * weight_class_prior['spread']
    
    z_score = (observed - median) / spread
    return np.clip(z_score, -7, 7)


# Pre-calculate weight class priors once
print("Calculating weight class priors...")
weight_classes = ['Heavyweight', 'Light Heavyweight', 'Middleweight', 'Welterweight',
                 'Lightweight', 'Featherweight', 'Bantamweight', 'Flyweight']

stats_for_adjperf = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg', 'ctrl_time_per_min', 'kd_per_min']

wc_priors = {}
for wc in weight_classes:
    wc_priors[wc] = {}
    for stat in stats_for_adjperf:
        prior = calculate_weight_class_prior(wc, stat, use_std=True)
        if prior:
            wc_priors[wc][stat] = prior

print(f"Done. Priors for {len(weight_classes)} weight classes × {len(stats_for_adjperf)} stats")

Calculating weight class priors...
Done. Priors for 8 weight classes × 7 stats


In [22]:
# ============================================================
# CELL 11 (continued): Generate AdjPerf Features
# ============================================================

def calculate_fighter_adjperf_rating(fighter_id, stat_name, as_of_date, weight_class, window=5):
    """
    Calculate fighter's AdjPerf rating by:
    1. Get their last N fights
    2. For each fight, calculate opponent-adjusted z-score
    3. Time-decay average those z-scores
    """
    
    if fighter_id not in fighter_fights_dict:
        return 0
    
    fights = fighter_fights_dict[fighter_id]
    recent = fights[fights['event_date'] < as_of_date].tail(window)
    
    if len(recent) == 0:
        return 0
    
    # Get weight class prior
    wc = 'Welterweight'  # default
    for wc_name in weight_classes:
        if wc_name in str(weight_class):
            wc = wc_name
            break
    
    wc_prior = wc_priors.get(wc, {}).get(stat_name)
    if wc_prior is None:
        return 0
    
    # Calculate z-score for each fight
    z_scores = []
    fight_dates = []
    
    for _, fight in recent.iterrows():
        observed = fight[stat_name]
        fight_date = fight['event_date']
        fight_id = fight['fight_id']
        
        # Get opponent ID
        opp_id = opponents_dict.get((fight_id, fighter_id))
        if opp_id is None:
            continue
        
        # Get opponent's baseline (what they typically allow)
        opp_baseline = calculate_opponent_adjperf_baseline(opp_id, stat_name, fight_date, window=10, use_std=True)
        
        # Calculate z-score
        z = calculate_adjperf_zscore(observed, opp_baseline, wc_prior, shrinkage_k=5)
        z_scores.append(z)
        fight_dates.append(fight_date)
    
    if len(z_scores) == 0:
        return 0
    
    # Time-decay average the z-scores
    weights = time_decay_weights(fight_dates, as_of_date, LAM)
    return np.average(z_scores, weights=weights)


def generate_adjperf_features(base_df, stats=stats_for_adjperf, window=5):
    """Generate AdjPerf features for all fights"""
    
    rows = []
    
    for idx, fight in base_df.iterrows():
        if idx % 500 == 0:
            print(f"  Processing {idx}/{len(base_df)}...")
        
        row = {'fight_id': fight['fight_id']}
        
        for stat in stats:
            f1_adjperf = calculate_fighter_adjperf_rating(
                fight['fighter_1_id'], stat, fight['event_date'], fight['weight_class'], window
            )
            f2_adjperf = calculate_fighter_adjperf_rating(
                fight['fighter_2_id'], stat, fight['event_date'], fight['weight_class'], window
            )
            
            row[f'f1_{stat}_adjperf'] = f1_adjperf
            row[f'f2_{stat}_adjperf'] = f2_adjperf
            row[f'diff_{stat}_adjperf'] = f1_adjperf - f2_adjperf
        
        rows.append(row)
    
    adjperf_df = pd.DataFrame(rows)
    return base_df.merge(adjperf_df, on='fight_id', how='left')


# Generate
print("Generating AdjPerf features...")
start = time.time()
fight_features_adjperf = generate_adjperf_features(fight_features, stats=stats_for_adjperf, window=5)
print(f"Done in {time.time()-start:.1f}s")
print(f"Shape: {fight_features_adjperf.shape}")

Generating AdjPerf features...
  Processing 0/7800...
  Processing 500/7800...
  Processing 1000/7800...
  Processing 1500/7800...
  Processing 2000/7800...
  Processing 2500/7800...
  Processing 3000/7800...
  Processing 3500/7800...
  Processing 4000/7800...
  Processing 4500/7800...
  Processing 5000/7800...
  Processing 5500/7800...
  Processing 6000/7800...
  Processing 6500/7800...
  Processing 7000/7800...
  Processing 7500/7800...
Done in 207.0s
Shape: (7800, 73)


In [42]:
# Update stats list for AdjPerf
stats_for_adjperf = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg', 'ctrl_time_per_min', 'kd_per_min',
                     'head_slpm', 'body_slpm', 'leg_slpm', 'distance_slpm', 'clinch_slpm', 'ground_slpm',
                     'head_acc', 'body_acc', 'leg_acc', 'distance_acc', 'clinch_acc', 'ground_acc']

# Recalculate weight class priors
print("Recalculating priors with accuracy stats...")
wc_priors = {}
for wc in weight_classes:
    wc_priors[wc] = {}
    for stat in stats_for_adjperf:
        prior = calculate_weight_class_prior(wc, stat, use_std=True)
        if prior:
            wc_priors[wc][stat] = prior

print(f"Done. Priors for {len(weight_classes)} × {len(stats_for_adjperf)} stats")

# Generate AdjPerf
print("Generating AdjPerf with accuracy stats...")
start = time.time()
fight_features_adjperf = generate_adjperf_features(fight_features, stats=stats_for_adjperf, window=5)
print(f"Done in {time.time()-start:.1f}s")

# Add UFC age
print("Adding UFC age...")
fight_features_adjperf = add_ufc_age_features(fight_features_adjperf)

# Add head defense
print("Adding head defense...")
fight_features_adjperf = add_head_defense_features(fight_features_adjperf)

print(f"Final shape: {fight_features_adjperf.shape}")

Recalculating priors with accuracy stats...
Done. Priors for 8 × 19 stats
Generating AdjPerf with accuracy stats...
  Processing 0/7800...
  Processing 500/7800...
  Processing 1000/7800...
  Processing 1500/7800...
  Processing 2000/7800...
  Processing 2500/7800...
  Processing 3000/7800...
  Processing 3500/7800...
  Processing 4000/7800...
  Processing 4500/7800...
  Processing 5000/7800...
  Processing 5500/7800...
  Processing 6000/7800...
  Processing 6500/7800...
  Processing 7000/7800...
  Processing 7500/7800...
Done in 541.9s
Adding UFC age...
  Processing 0/7800...
  Processing 1000/7800...
  Processing 2000/7800...
  Processing 3000/7800...
  Processing 4000/7800...
  Processing 5000/7800...
  Processing 6000/7800...
  Processing 7000/7800...
Adding head defense...
  Processing 0/7800...
  Processing 1000/7800...
  Processing 2000/7800...
  Processing 3000/7800...
  Processing 4000/7800...
  Processing 5000/7800...
  Processing 6000/7800...
  Processing 7000/7800...
Final 

In [32]:
# ============================================================
# CELL 10: Model Test with Strike Breakdown
# ============================================================

drop_cols = ['fight_id', 'event_date', 'event_name', 'weight_class', 'method',
             'ending_round', 'ending_time', 'fighter_1_id', 'fighter_1_name',
             'fighter_2_id', 'fighter_2_name', 'winner',
             'f1_reach', 'f2_reach', 'f1_height', 'f2_height', 'f1_age', 'f2_age']

# Only use diff + physical features
keep_cols = [c for c in fight_features_adjperf.columns 
             if c.startswith('diff_') 
             or c in ['age_diff', 'reach_diff', 'height_diff',
                      'age_ratio', 'reach_ratio', 'height_ratio', 'stance_matchup']]

train_data = fight_features_adjperf[fight_features_adjperf['event_date'] < '2021-01-01']
val_data   = fight_features_adjperf[fight_features_adjperf['event_date'] >= '2021-01-01']

X_train = train_data[keep_cols].fillna(0)
y_train = train_data['winner']
X_val   = val_data[keep_cols].fillna(0)
y_val   = val_data['winner']

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Features: {len(keep_cols)}")

model = XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    min_child_weight=8, subsample=0.7, colsample_bytree=0.7,
    gamma=1.0, random_state=42
)

model.fit(X_train, y_train)
print(f"\nPrevious (without strike breakdown): 58.8%")
print(f"Training:   {model.score(X_train, y_train):.1%}")
print(f"Validation: {model.score(X_val, y_val):.1%}")

# Check if head_slpm made it to top features
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 features:")
print(importance_df.head(20).to_string())

Train: 6654 | Val: 1146 | Features: 35

Previous (without strike breakdown): 58.8%
Training:   70.5%
Validation: 57.4%

Top 20 features:
                     feature  importance
20                 age_ratio    0.046316
1               diff_str_acc    0.042716
17                  age_diff    0.042199
13         diff_total_fights    0.040735
3                diff_td_avg    0.038827
12          diff_ground_slpm    0.035713
18               reach_ratio    0.034445
7             diff_head_slpm    0.032924
0                  diff_slpm    0.030618
6            diff_kd_per_min    0.029438
10        diff_distance_slpm    0.029194
4               diff_sub_avg    0.027995
5     diff_ctrl_time_per_min    0.027295
2                diff_td_acc    0.027077
28   diff_kd_per_min_adjperf    0.026985
8             diff_body_slpm    0.026896
34  diff_ground_slpm_adjperf    0.026400
33  diff_clinch_slpm_adjperf    0.026055
25       diff_td_avg_adjperf    0.025305
14      diff_days_since_last    0.025245


In [33]:
# Feature importance with AdjPerf
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 features:")
print(importance_df.head(20).to_string())

# Count how many AdjPerf features in top 20
adjperf_in_top20 = importance_df.head(20)['feature'].str.contains('adjperf').sum()
print(f"\nAdjPerf features in top 20: {adjperf_in_top20}")

Top 20 features:
                     feature  importance
20                 age_ratio    0.046316
1               diff_str_acc    0.042716
17                  age_diff    0.042199
13         diff_total_fights    0.040735
3                diff_td_avg    0.038827
12          diff_ground_slpm    0.035713
18               reach_ratio    0.034445
7             diff_head_slpm    0.032924
0                  diff_slpm    0.030618
6            diff_kd_per_min    0.029438
10        diff_distance_slpm    0.029194
4               diff_sub_avg    0.027995
5     diff_ctrl_time_per_min    0.027295
2                diff_td_acc    0.027077
28   diff_kd_per_min_adjperf    0.026985
8             diff_body_slpm    0.026896
34  diff_ground_slpm_adjperf    0.026400
33  diff_clinch_slpm_adjperf    0.026055
25       diff_td_avg_adjperf    0.025305
14      diff_days_since_last    0.025245

AdjPerf features in top 20: 4


In [30]:
# Check which strike breakdown features are actually top 20
strike_features = importance_df[importance_df['feature'].str.contains('head|body|leg|distance|clinch|ground')]
print("\nStrike breakdown features ranking:")
print(strike_features.head(15).to_string())


Strike breakdown features ranking:
                       feature  importance
12            diff_ground_slpm    0.035713
7               diff_head_slpm    0.032924
10          diff_distance_slpm    0.029194
8               diff_body_slpm    0.026896
34    diff_ground_slpm_adjperf    0.026400
33    diff_clinch_slpm_adjperf    0.026055
29      diff_head_slpm_adjperf    0.025196
11            diff_clinch_slpm    0.024248
30      diff_body_slpm_adjperf    0.023854
31       diff_leg_slpm_adjperf    0.023721
9                diff_leg_slpm    0.023364
32  diff_distance_slpm_adjperf    0.023306


In [34]:
# ============================================================
# CELL 13: UFC Age (Time in Promotion)
# ============================================================

def calculate_ufc_age(fighter_id, as_of_date):
    """
    Calculate how long fighter has been in UFC (days since first UFC fight).
    """
    if fighter_id not in fighter_fights_dict:
        return 0
    
    all_fights = fighter_fights_dict[fighter_id]
    ufc_fights = all_fights[all_fights['event_date'] < as_of_date]
    
    if len(ufc_fights) == 0:
        return 0
    
    first_ufc_fight = datetime.strptime(ufc_fights.iloc[0]['event_date'], "%Y-%m-%d")
    current_date = datetime.strptime(as_of_date, "%Y-%m-%d")
    
    ufc_age_days = (current_date - first_ufc_fight).days
    return ufc_age_days / 365.25  # Convert to years


def add_ufc_age_features(df):
    """Add UFC age for both fighters + difference"""
    
    rows = []
    
    for idx, fight in df.iterrows():
        if idx % 1000 == 0:
            print(f"  Processing {idx}/{len(df)}...")
        
        f1_ufc_age = calculate_ufc_age(fight['fighter_1_id'], fight['event_date'])
        f2_ufc_age = calculate_ufc_age(fight['fighter_2_id'], fight['event_date'])
        
        rows.append({
            'fight_id': fight['fight_id'],
            'f1_ufc_age': f1_ufc_age,
            'f2_ufc_age': f2_ufc_age,
            'diff_ufc_age': f1_ufc_age - f2_ufc_age
        })
    
    ufc_age_df = pd.DataFrame(rows)
    return df.merge(ufc_age_df, on='fight_id', how='left')


# Add to existing features
print("Adding UFC Age features...")
start = time.time()
fight_features_adjperf = add_ufc_age_features(fight_features_adjperf)
print(f"Done in {time.time()-start:.1f}s")
print(f"Shape: {fight_features_adjperf.shape}")

Adding UFC Age features...
  Processing 0/7800...
  Processing 1000/7800...
  Processing 2000/7800...
  Processing 3000/7800...
  Processing 4000/7800...
  Processing 5000/7800...
  Processing 6000/7800...
  Processing 7000/7800...
Done in 5.4s
Shape: (7800, 112)


In [35]:
# Update keep_cols to include diff_ufc_age
keep_cols = [c for c in fight_features_adjperf.columns 
             if c.startswith('diff_') 
             or c in ['age_diff', 'reach_diff', 'height_diff',
                      'age_ratio', 'reach_ratio', 'height_ratio', 'stance_matchup']]

train_data = fight_features_adjperf[fight_features_adjperf['event_date'] < '2021-01-01']
val_data   = fight_features_adjperf[fight_features_adjperf['event_date'] >= '2021-01-01']

X_train = train_data[keep_cols].fillna(0)
y_train = train_data['winner']
X_val   = val_data[keep_cols].fillna(0)
y_val   = val_data['winner']

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Features: {len(keep_cols)}")

model = XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    min_child_weight=8, subsample=0.7, colsample_bytree=0.7,
    gamma=1.0, random_state=42
)

model.fit(X_train, y_train)
print(f"\nPrevious: 57.4%")
print(f"Training:   {model.score(X_train, y_train):.1%}")
print(f"Validation: {model.score(X_val, y_val):.1%}")

# Check diff_ufc_age ranking
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

ufc_age_rank = importance_df[importance_df['feature'] == 'diff_ufc_age']
print(f"\ndiff_ufc_age ranking: {ufc_age_rank.index[0]+1 if len(ufc_age_rank) > 0 else 'Not found'}")
print(f"Importance: {ufc_age_rank['importance'].iloc[0]:.4f}" if len(ufc_age_rank) > 0 else "")

Train: 6654 | Val: 1146 | Features: 36

Previous: 57.4%
Training:   70.9%
Validation: 58.0%

diff_ufc_age ranking: 36
Importance: 0.0282


In [36]:
# ============================================================
# CELL 14: Head Defense (Strikes Absorbed)
# ============================================================

def calculate_head_defense_stats(fighter_id, as_of_date, window=5):
    """
    Calculate how many head strikes this fighter ABSORBS per minute.
    Lower is better (good defense).
    """
    if fighter_id not in fighter_fights_dict:
        return None
    
    fights = fighter_fights_dict[fighter_id]
    recent = fights[fights['event_date'] < as_of_date].tail(window)
    
    if len(recent) == 0:
        return None
    
    weights = time_decay_weights(recent['event_date'].tolist(), as_of_date, LAM)
    
    # For each fight, get opponent's head strikes against this fighter
    head_absorbed_rates = []
    
    for _, fight in recent.iterrows():
        fight_id = fight['fight_id']
        
        # Get opponent
        opp_id = opponents_dict.get((fight_id, fighter_id))
        if opp_id is None:
            continue
        
        # Get opponent's performance in this fight
        opp_fight = fighter_fights_dict.get(opp_id)
        if opp_fight is None:
            continue
        
        opp_this_fight = opp_fight[opp_fight['fight_id'] == fight_id]
        if len(opp_this_fight) == 0:
            continue
        
        # Opponent's head strikes landed (absorbed by our fighter)
        head_absorbed = opp_this_fight['head_slpm'].iloc[0]
        head_absorbed_rates.append(head_absorbed)
    
    if len(head_absorbed_rates) == 0:
        return None
    
    # Time-decay weighted average
    weighted_head_absorbed = np.average(head_absorbed_rates, weights=weights[:len(head_absorbed_rates)])
    
    return {
        'head_absorbed_pm': weighted_head_absorbed
    }


def add_head_defense_features(df):
    rows = []
    
    for idx, fight in df.iterrows():
        if idx % 1000 == 0:
            print(f"  Processing {idx}/{len(df)}...")
        
        f1_def = calculate_head_defense_stats(fight['fighter_1_id'], fight['event_date'])
        f2_def = calculate_head_defense_stats(fight['fighter_2_id'], fight['event_date'])
        
        f1_absorbed = f1_def['head_absorbed_pm'] if f1_def else 0
        f2_absorbed = f2_def['head_absorbed_pm'] if f2_def else 0
        
        rows.append({
            'fight_id': fight['fight_id'],
            'f1_head_absorbed_pm': f1_absorbed,
            'f2_head_absorbed_pm': f2_absorbed,
            'diff_head_absorbed_pm': f1_absorbed - f2_absorbed  # Higher = worse defense
        })
    
    defense_df = pd.DataFrame(rows)
    return df.merge(defense_df, on='fight_id', how='left')


print("Adding head defense features...")
start = time.time()
fight_features_adjperf = add_head_defense_features(fight_features_adjperf)
print(f"Done in {time.time()-start:.1f}s")
print(f"Shape: {fight_features_adjperf.shape}")

Adding head defense features...
  Processing 0/7800...
  Processing 1000/7800...
  Processing 2000/7800...
  Processing 3000/7800...
  Processing 4000/7800...
  Processing 5000/7800...
  Processing 6000/7800...
  Processing 7000/7800...
Done in 21.4s
Shape: (7800, 115)


In [37]:
keep_cols = [c for c in fight_features_adjperf.columns 
             if c.startswith('diff_') 
             or c in ['age_diff', 'reach_diff', 'height_diff',
                      'age_ratio', 'reach_ratio', 'height_ratio', 'stance_matchup']]

train_data = fight_features_adjperf[fight_features_adjperf['event_date'] < '2021-01-01']
val_data   = fight_features_adjperf[fight_features_adjperf['event_date'] >= '2021-01-01']

X_train = train_data[keep_cols].fillna(0)
y_train = train_data['winner']
X_val   = val_data[keep_cols].fillna(0)
y_val   = val_data['winner']

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Features: {len(keep_cols)}")

model = XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    min_child_weight=8, subsample=0.7, colsample_bytree=0.7,
    gamma=1.0, random_state=42
)

model.fit(X_train, y_train)
print(f"\nPrevious: 58.0%")
print(f"Training:   {model.score(X_train, y_train):.1%}")
print(f"Validation: {model.score(X_val, y_val):.1%}")

# Check head defense ranking
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 features:")
print(importance_df.head(15).to_string())

Train: 6654 | Val: 1146 | Features: 37

Previous: 58.0%
Training:   71.0%
Validation: 59.1%

Top 15 features:
                  feature  importance
17               age_diff    0.045889
3             diff_td_avg    0.043700
20              age_ratio    0.042286
1            diff_str_acc    0.037078
7          diff_head_slpm    0.034699
13      diff_total_fights    0.033836
18            reach_ratio    0.032904
12       diff_ground_slpm    0.029907
36  diff_head_absorbed_pm    0.029441
2             diff_td_acc    0.027759
15             reach_diff    0.027656
35           diff_ufc_age    0.027378
6         diff_kd_per_min    0.026759
9           diff_leg_slpm    0.026721
0               diff_slpm    0.026675


In [38]:
# ============================================================
# Validation: Verify Feature Calculations
# ============================================================

# Pick Jon Jones vs Thiago Santos (2019-07-06)
test_fight = fight_features_adjperf[
    (fight_features_adjperf['fighter_1_name'].str.contains('Jon Jones', na=False)) &
    (fight_features_adjperf['event_date'] == '2019-07-06')
]

jones_id = test_fight['fighter_1_id'].iloc[0]
santos_id = test_fight['fighter_2_id'].iloc[0]
fight_date = '2019-07-06'

print("="*60)
print("FEATURE VALIDATION: Jon Jones vs Thiago Santos (2019-07-06)")
print("="*60)

# 1. VERIFY DECAYED ROLLING STATS
print("\n1. DECAYED ROLLING STATS")
print("-" * 60)

jones_history = fighter_fights_dict[jones_id]
jones_recent = jones_history[jones_history['event_date'] < fight_date].tail(5)

print("Jon Jones last 5 fights:")
print(jones_recent[['event_date', 'slpm', 'head_slpm']].to_string())

weights = time_decay_weights(jones_recent['event_date'].tolist(), fight_date, LAM)
print(f"\nDecay weights: {weights.round(3)}")

manual_slpm = np.average(jones_recent['slpm'].values, weights=weights)
manual_head = np.average(jones_recent['head_slpm'].values, weights=weights)

print(f"\nManual calculation:")
print(f"  slpm (decayed avg): {manual_slpm:.3f}")
print(f"  head_slpm (decayed avg): {manual_head:.3f}")

print(f"\nDataset values:")
print(f"  f1_slpm: {test_fight['f1_slpm'].iloc[0]:.3f}")
print(f"  f1_head_slpm: {test_fight['f1_head_slpm'].iloc[0]:.3f}")

print(f"\nMatch: {abs(manual_slpm - test_fight['f1_slpm'].iloc[0]) < 0.01}")

# 2. VERIFY PHYSICAL FEATURES
print("\n2. PHYSICAL FEATURES")
print("-" * 60)

jones_fighter = pd.read_sql(f"""
    SELECT reach, height, dob FROM fighters_v2 WHERE fighter_id = '{jones_id}'
""", conn)

jones_reach = parse_reach(jones_fighter['reach'].iloc[0])
jones_age = parse_age(jones_fighter['dob'].iloc[0], fight_date)

print(f"Jon Jones:")
print(f"  Reach: {jones_reach}\"")
print(f"  Age at fight: {jones_age:.1f} years")

print(f"\nDataset values:")
print(f"  f1_reach: {test_fight['f1_reach'].iloc[0]}")
print(f"  f1_age: {test_fight['f1_age'].iloc[0]:.1f}")

print(f"\nMatch: {abs(jones_reach - test_fight['f1_reach'].iloc[0]) < 0.1}")

# 3. VERIFY UFC AGE
print("\n3. UFC AGE")
print("-" * 60)

jones_first_fight = jones_history.iloc[0]['event_date']
ufc_age_manual = (datetime.strptime(fight_date, "%Y-%m-%d") - 
                  datetime.strptime(jones_first_fight, "%Y-%m-%d")).days / 365.25

print(f"First UFC fight: {jones_first_fight}")
print(f"UFC age (manual): {ufc_age_manual:.2f} years")
print(f"UFC age (dataset): {test_fight['f1_ufc_age'].iloc[0]:.2f} years")
print(f"Match: {abs(ufc_age_manual - test_fight['f1_ufc_age'].iloc[0]) < 0.01}")

# 4. VERIFY HEAD DEFENSE
print("\n4. HEAD DEFENSE (ABSORBED)")
print("-" * 60)

# Get opponents' head strikes against Jones
head_absorbed_list = []
for _, fight in jones_recent.iterrows():
    opp_id = opponents_dict.get((fight['fight_id'], jones_id))
    if opp_id and opp_id in fighter_fights_dict:
        opp_fight = fighter_fights_dict[opp_id]
        opp_this_fight = opp_fight[opp_fight['fight_id'] == fight['fight_id']]
        if len(opp_this_fight) > 0:
            head_absorbed_list.append(opp_this_fight['head_slpm'].iloc[0])

if head_absorbed_list:
    manual_head_absorbed = np.average(head_absorbed_list, weights=weights[:len(head_absorbed_list)])
    print(f"Opponents' head strikes landed against Jones: {[f'{x:.2f}' for x in head_absorbed_list]}")
    print(f"Manual calculation: {manual_head_absorbed:.3f}")
    print(f"Dataset value: {test_fight['f1_head_absorbed_pm'].iloc[0]:.3f}")
    print(f"Match: {abs(manual_head_absorbed - test_fight['f1_head_absorbed_pm'].iloc[0]) < 0.01}")

print("\n" + "="*60)
print("VALIDATION COMPLETE")
print("="*60)

FEATURE VALIDATION: Jon Jones vs Thiago Santos (2019-07-06)

1. DECAYED ROLLING STATS
------------------------------------------------------------
Jon Jones last 5 fights:
    event_date      slpm  head_slpm
15  2015-01-03  3.680000   1.720000
16  2016-04-23  4.200000   1.400000
17  2017-07-29  7.298335   2.535211
18  2018-12-29  4.903047   2.160665
19  2019-03-02  5.000000   1.800000

Decay weights: [0.052 0.095 0.17  0.328 0.355]

Manual calculation:
  slpm (decayed avg): 5.215
  head_slpm (decayed avg): 2.001

Dataset values:
  f1_slpm: 5.215
  f1_head_slpm: 2.001

Match: True

2. PHYSICAL FEATURES
------------------------------------------------------------
Jon Jones:
  Reach: 84.0"
  Age at fight: 32.0 years

Dataset values:
  f1_reach: 84.0
  f1_age: 32.0

Match: True

3. UFC AGE
------------------------------------------------------------
First UFC fight: 2008-08-09
UFC age (manual): 10.90 years
UFC age (dataset): 10.90 years
Match: True

4. HEAD DEFENSE (ABSORBED)
-------------

In [39]:
# Check what columns are available in sig_strikes table
sample_sig = pd.read_sql("""
    SELECT * FROM fight_round_sig_strikes_v2 LIMIT 2
""", conn)

print(sample_sig.columns.tolist())


['round_id', 'fighter_id', 'head_landed', 'head_attempted', 'body_landed', 'body_attempted', 'leg_landed', 'leg_attempted', 'distance_landed', 'distance_attempted', 'clinch_landed', 'clinch_attempted', 'ground_landed', 'ground_attempted']


In [40]:
# ============================================================
# CELL 4 (Updated): Add Strike Accuracy Stats
# ============================================================

print("Fetching all fight stats with strike breakdown (landed + attempted)...")
start = time.time()

# Original fight totals
all_fight_stats = pd.read_sql("""
    SELECT 
        f.fight_id,
        f.event_date,
        f.weight_class,
        f.ending_round,
        f.ending_time,
        f.method,
        ff.fighter_id,
        SUM(frs.sig_str_landed) as sig_str_landed,
        SUM(frs.sig_str_attempted) as sig_str_attempted,
        SUM(frs.td_landed) as td_landed,
        SUM(frs.td_attempted) as td_attempted,
        SUM(frs.sub_attempts) as sub_attempts,
        SUM(frs.ctrl_seconds) as ctrl_seconds,
        SUM(frs.knockdowns) as knockdowns,
        COUNT(DISTINCT fr.round_number) as total_rounds
    FROM fight_round_stats_v2 frs
    JOIN fight_rounds_v2 fr ON frs.round_id = fr.round_id
    JOIN fights_v2 f ON fr.fight_id = f.fight_id
    JOIN fight_fighters_v2 ff ON f.fight_id = ff.fight_id AND frs.fighter_id = ff.fighter_id
    GROUP BY f.fight_id, ff.fighter_id
""", conn)

# Strike breakdown (landed + attempted)
strike_breakdown = pd.read_sql("""
    SELECT 
        fr.fight_id,
        frss.fighter_id,
        SUM(frss.head_landed) as head_landed,
        SUM(frss.head_attempted) as head_attempted,
        SUM(frss.body_landed) as body_landed,
        SUM(frss.body_attempted) as body_attempted,
        SUM(frss.leg_landed) as leg_landed,
        SUM(frss.leg_attempted) as leg_attempted,
        SUM(frss.distance_landed) as distance_landed,
        SUM(frss.distance_attempted) as distance_attempted,
        SUM(frss.clinch_landed) as clinch_landed,
        SUM(frss.clinch_attempted) as clinch_attempted,
        SUM(frss.ground_landed) as ground_landed,
        SUM(frss.ground_attempted) as ground_attempted
    FROM fight_round_sig_strikes_v2 frss
    JOIN fight_rounds_v2 fr ON frss.round_id = fr.round_id
    GROUP BY fr.fight_id, frss.fighter_id
""", conn)

all_fight_stats = all_fight_stats.merge(strike_breakdown, on=['fight_id', 'fighter_id'], how='left').fillna(0)

all_fight_stats['actual_minutes'] = all_fight_stats.apply(
    lambda r: parse_fight_duration(r['ending_round'], r['ending_time']), axis=1
)

# Original stats
all_fight_stats['slpm']             = all_fight_stats['sig_str_landed'] / all_fight_stats['actual_minutes']
all_fight_stats['str_acc']          = all_fight_stats['sig_str_landed'] / all_fight_stats['sig_str_attempted']
all_fight_stats['td_acc']           = all_fight_stats['td_landed'] / all_fight_stats['td_attempted']
all_fight_stats['td_avg']           = all_fight_stats['td_landed'] / (all_fight_stats['actual_minutes'] / 15)
all_fight_stats['sub_avg']          = all_fight_stats['sub_attempts'] / (all_fight_stats['actual_minutes'] / 15)
all_fight_stats['ctrl_time_per_min']= all_fight_stats['ctrl_seconds'] / all_fight_stats['actual_minutes'] / 60
all_fight_stats['kd_per_min']       = all_fight_stats['knockdowns'] / all_fight_stats['actual_minutes']

# Strike volume (per minute)
all_fight_stats['head_slpm']     = all_fight_stats['head_landed'] / all_fight_stats['actual_minutes']
all_fight_stats['body_slpm']     = all_fight_stats['body_landed'] / all_fight_stats['actual_minutes']
all_fight_stats['leg_slpm']      = all_fight_stats['leg_landed'] / all_fight_stats['actual_minutes']
all_fight_stats['distance_slpm'] = all_fight_stats['distance_landed'] / all_fight_stats['actual_minutes']
all_fight_stats['clinch_slpm']   = all_fight_stats['clinch_landed'] / all_fight_stats['actual_minutes']
all_fight_stats['ground_slpm']   = all_fight_stats['ground_landed'] / all_fight_stats['actual_minutes']

# Strike accuracy (NEW)
all_fight_stats['head_acc']      = all_fight_stats['head_landed'] / all_fight_stats['head_attempted']
all_fight_stats['body_acc']      = all_fight_stats['body_landed'] / all_fight_stats['body_attempted']
all_fight_stats['leg_acc']       = all_fight_stats['leg_landed'] / all_fight_stats['leg_attempted']
all_fight_stats['distance_acc']  = all_fight_stats['distance_landed'] / all_fight_stats['distance_attempted']
all_fight_stats['clinch_acc']    = all_fight_stats['clinch_landed'] / all_fight_stats['clinch_attempted']
all_fight_stats['ground_acc']    = all_fight_stats['ground_landed'] / all_fight_stats['ground_attempted']

all_fight_stats = all_fight_stats.replace([np.inf, -np.inf], np.nan).fillna(0)
all_fight_stats = all_fight_stats.sort_values('event_date').reset_index(drop=True)

all_opponents = pd.read_sql("""
    SELECT ff1.fight_id, ff1.fighter_id, ff2.fighter_id as opponent_id
    FROM fight_fighters_v2 ff1
    JOIN fight_fighters_v2 ff2 ON ff1.fight_id = ff2.fight_id AND ff1.fighter_id != ff2.fighter_id
""", conn)

fighter_fights_dict = {
    fid: group.sort_values('event_date').reset_index(drop=True)
    for fid, group in all_fight_stats.groupby('fighter_id')
}

opponents_dict = {
    (row['fight_id'], row['fighter_id']): row['opponent_id']
    for _, row in all_opponents.iterrows()
}

print(f"Done in {time.time()-start:.1f}s")
print(f"Fighters: {len(fighter_fights_dict)} | Opponents: {len(opponents_dict)}")
print(f"New accuracy stats: head_acc, body_acc, leg_acc, distance_acc, clinch_acc, ground_acc")

Fetching all fight stats with strike breakdown (landed + attempted)...
Done in 4.1s
Fighters: 3760 | Opponents: 22254
New accuracy stats: head_acc, body_acc, leg_acc, distance_acc, clinch_acc, ground_acc


In [43]:
# ============================================================
# Model Test with Accuracy Stats
# ============================================================

keep_cols = [c for c in fight_features_adjperf.columns 
             if c.startswith('diff_') 
             or c in ['age_diff', 'reach_diff', 'height_diff',
                      'age_ratio', 'reach_ratio', 'height_ratio', 'stance_matchup']]

train_data = fight_features_adjperf[fight_features_adjperf['event_date'] < '2021-01-01']
val_data   = fight_features_adjperf[fight_features_adjperf['event_date'] >= '2021-01-01']

X_train = train_data[keep_cols].fillna(0)
y_train = train_data['winner']
X_val   = val_data[keep_cols].fillna(0)
y_val   = val_data['winner']

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Features: {len(keep_cols)}")

model = XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    min_child_weight=8, subsample=0.7, colsample_bytree=0.7,
    gamma=1.0, random_state=42
)

model.fit(X_train, y_train)
print(f"\nPrevious: 59.1%")
print(f"Training:   {model.score(X_train, y_train):.1%}")
print(f"Validation: {model.score(X_val, y_val):.1%}")

# Check top accuracy features
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

acc_features = importance_df[importance_df['feature'].str.contains('_acc') & ~importance_df['feature'].str.contains('str_acc')]
print("\nAccuracy feature rankings:")
print(acc_features.head(10).to_string())

Train: 6654 | Val: 1146 | Features: 49

Previous: 59.1%
Training:   72.3%
Validation: 58.9%

Accuracy feature rankings:
                      feature  importance
13              diff_head_acc    0.031290
16          diff_distance_acc    0.024489
2                 diff_td_acc    0.021666
15               diff_leg_acc    0.021658
41      diff_head_acc_adjperf    0.019561
18            diff_ground_acc    0.019081
42      diff_body_acc_adjperf    0.018960
14              diff_body_acc    0.018008
17            diff_clinch_acc    0.017300
44  diff_distance_acc_adjperf    0.017100


In [44]:
# Keep only top 30 features
top_30 = importance_df.head(30)['feature'].tolist()

X_train_reduced = X_train[top_30]
X_val_reduced = X_val[top_30]

model.fit(X_train_reduced, y_train)
print(f"With top 30 features:")
print(f"Training:   {model.score(X_train_reduced, y_train):.1%}")
print(f"Validation: {model.score(X_val_reduced, y_val):.1%}")

With top 30 features:
Training:   70.8%
Validation: 58.8%


In [45]:
# ============================================================
# CELL 15: Round-Specific Features
# ============================================================

print("Pre-fetching all round-level stats...")
start = time.time()

all_rounds = pd.read_sql("""
    SELECT 
        f.fight_id,
        f.event_date,
        ff.fighter_id,
        fr.round_number,
        frs.sig_str_landed,
        frs.sig_str_attempted,
        frs.td_landed,
        frs.td_attempted,
        frs.ctrl_seconds,
        frs.knockdowns
    FROM fight_round_stats_v2 frs
    JOIN fight_rounds_v2 fr ON frs.round_id = fr.round_id
    JOIN fights_v2 f ON fr.fight_id = f.fight_id
    JOIN fight_fighters_v2 ff ON f.fight_id = ff.fight_id AND frs.fighter_id = ff.fighter_id
    ORDER BY f.event_date, fr.round_number
""", conn)

# Calculate per-minute stats for each round (each round = 5 min)
all_rounds['slpm'] = all_rounds['sig_str_landed'] / 5
all_rounds['str_acc'] = all_rounds['sig_str_landed'] / all_rounds['sig_str_attempted']
all_rounds['td_per_min'] = all_rounds['td_landed'] / 5
all_rounds['ctrl_per_min'] = all_rounds['ctrl_seconds'] / 5 / 60
all_rounds['kd_per_min'] = all_rounds['knockdowns'] / 5

all_rounds = all_rounds.replace([np.inf, -np.inf], np.nan).fillna(0)

# Build dict for O(1) lookups
fighter_rounds_dict = {
    fid: group.sort_values(['event_date', 'round_number'])
    for fid, group in all_rounds.groupby('fighter_id')
}

print(f"Done in {time.time()-start:.1f}s")
print(f"Fighters with round data: {len(fighter_rounds_dict)}")


def calculate_round_features(fighter_id, as_of_date, window=10):
    """
    Calculate Round 1 specialization and cardio indicators.
    """
    
    if fighter_id not in fighter_rounds_dict:
        return None
    
    rounds = fighter_rounds_dict[fighter_id]
    recent_fights = rounds[rounds['event_date'] < as_of_date]['fight_id'].unique()[-window:]
    recent = rounds[rounds['fight_id'].isin(recent_fights)]
    
    if len(recent) == 0:
        return None
    
    # Overall average
    overall_slpm = recent['slpm'].mean()
    overall_ctrl = recent['ctrl_per_min'].mean()
    
    # Round 1 specific
    r1 = recent[recent['round_number'] == 1]
    r1_slpm = r1['slpm'].mean() if len(r1) > 0 else 0
    r1_ctrl = r1['ctrl_per_min'].mean() if len(r1) > 0 else 0
    
    # Round 3+ (cardio)
    r3plus = recent[recent['round_number'] >= 3]
    r3_slpm = r3plus['slpm'].mean() if len(r3plus) > 0 else 0
    r3_ctrl = r3plus['ctrl_per_min'].mean() if len(r3plus) > 0 else 0
    
    # Fast starter indicator (R1 vs overall)
    fast_starter = (r1_slpm / overall_slpm) if overall_slpm > 0 else 1.0
    
    # Cardio fade indicator (R3+ vs R1)
    cardio_ratio = (r3_slpm / r1_slpm) if r1_slpm > 0 else 1.0
    
    return {
        'r1_slpm': r1_slpm,
        'r1_ctrl': r1_ctrl,
        'r3_slpm': r3_slpm,
        'r3_ctrl': r3_ctrl,
        'fast_starter': fast_starter,  # >1 = more active in R1
        'cardio_ratio': cardio_ratio   # <1 = fades, >1 = improves
    }


def add_round_features(df):
    rows = []
    
    for idx, fight in df.iterrows():
        if idx % 1000 == 0:
            print(f"  Processing {idx}/{len(df)}...")
        
        f1 = calculate_round_features(fight['fighter_1_id'], fight['event_date'])
        f2 = calculate_round_features(fight['fighter_2_id'], fight['event_date'])
        
        row = {'fight_id': fight['fight_id']}
        
        for key in ['r1_slpm', 'r1_ctrl', 'r3_slpm', 'r3_ctrl', 'fast_starter', 'cardio_ratio']:
            row[f'f1_{key}'] = f1[key] if f1 else 0
            row[f'f2_{key}'] = f2[key] if f2 else 0
            row[f'diff_{key}'] = row[f'f1_{key}'] - row[f'f2_{key}']
        
        rows.append(row)
    
    round_df = pd.DataFrame(rows)
    return df.merge(round_df, on='fight_id', how='left')


print("\nAdding round features...")
start = time.time()
fight_features_rounds = add_round_features(fight_features_adjperf)
print(f"Done in {time.time()-start:.1f}s")
print(f"Shape: {fight_features_rounds.shape}")

Pre-fetching all round-level stats...
Done in 2.8s
Fighters with round data: 3760

Adding round features...
  Processing 0/7800...
  Processing 1000/7800...
  Processing 2000/7800...
  Processing 3000/7800...
  Processing 4000/7800...
  Processing 5000/7800...
  Processing 6000/7800...
  Processing 7000/7800...
Done in 17.1s
Shape: (7800, 169)


In [46]:
# Test with round features
keep_cols = [c for c in fight_features_rounds.columns 
             if c.startswith('diff_') 
             or c in ['age_diff', 'reach_diff', 'height_diff',
                      'age_ratio', 'reach_ratio', 'height_ratio', 'stance_matchup']]

train_data = fight_features_rounds[fight_features_rounds['event_date'] < '2021-01-01']
val_data   = fight_features_rounds[fight_features_rounds['event_date'] >= '2021-01-01']

X_train = train_data[keep_cols].fillna(0)
y_train = train_data['winner']
X_val   = val_data[keep_cols].fillna(0)
y_val   = val_data['winner']

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Features: {len(keep_cols)}")

model = XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    min_child_weight=8, subsample=0.7, colsample_bytree=0.7,
    gamma=1.0, random_state=42
)

model.fit(X_train, y_train)
print(f"\nPrevious (without round features): 58.8%")
print(f"Training:   {model.score(X_train, y_train):.1%}")
print(f"Validation: {model.score(X_val, y_val):.1%}")

# Check round feature importance
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

round_features = importance_df[importance_df['feature'].str.contains('r1_|r3_|fast_starter|cardio')]
print("\nRound feature rankings:")
print(round_features.to_string())

Train: 6654 | Val: 1146 | Features: 55

Previous (without round features): 58.8%
Training:   72.2%
Validation: 59.5%

Round feature rankings:
              feature  importance
49       diff_r1_slpm    0.024573
50       diff_r1_ctrl    0.017326
51       diff_r3_slpm    0.016490
54  diff_cardio_ratio    0.015819
52       diff_r3_ctrl    0.015444
53  diff_fast_starter    0.014818


In [47]:
# Keep top 35 features
top_35 = importance_df.head(35)['feature'].tolist()

X_train_reduced = X_train[top_35]
X_val_reduced = X_val[top_35]

model.fit(X_train_reduced, y_train)
print(f"\nWith top 35 features:")
print(f"Training:   {model.score(X_train_reduced, y_train):.1%}")
print(f"Validation: {model.score(X_val_reduced, y_val):.1%}")

print(f"\nTop 35 features:")
print(importance_df.head(35)['feature'].tolist())


With top 35 features:
Training:   71.7%
Validation: 59.0%

Top 35 features:
['age_ratio', 'diff_head_acc', 'diff_str_acc', 'diff_total_fights', 'diff_td_avg', 'age_diff', 'diff_r1_slpm', 'diff_ground_slpm', 'reach_ratio', 'diff_head_slpm', 'reach_diff', 'diff_ground_acc', 'diff_distance_acc', 'diff_head_absorbed_pm', 'diff_leg_acc', 'diff_sub_avg', 'diff_slpm', 'diff_td_acc', 'diff_r1_ctrl', 'diff_clinch_acc', 'diff_ufc_age', 'diff_kd_per_min_adjperf', 'diff_distance_slpm', 'diff_body_slpm', 'diff_days_since_last', 'diff_body_acc_adjperf', 'diff_clinch_slpm_adjperf', 'diff_head_slpm_adjperf', 'diff_td_avg_adjperf', 'diff_r3_slpm', 'diff_kd_per_min', 'diff_body_slpm_adjperf', 'stance_matchup', 'diff_ground_acc_adjperf', 'diff_cardio_ratio']


In [48]:
# ============================================================
# Hyperparameter Optimization
# ============================================================

from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.03, 0.05, 0.1],
    'min_child_weight': [5, 8, 10],
    'subsample': [0.7, 0.8],
    'colsample_bytree': [0.7, 0.8],
    'gamma': [0.5, 1.0]
}

xgb_base = XGBClassifier(random_state=42)

grid_search = GridSearchCV(
    xgb_base, 
    param_grid, 
    cv=3,  # 3-fold CV
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

print("Running grid search (this will take ~20-30 minutes)...")
keep_cols = [c for c in fight_features_rounds.columns 
             if c.startswith('diff_') 
             or c in ['age_diff', 'reach_diff', 'height_diff',
                      'age_ratio', 'reach_ratio', 'height_ratio', 'stance_matchup']]

X_train_full = train_data[keep_cols].fillna(0)
grid_search.fit(X_train_full, y_train)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.1%}")

# Test on validation
best_model = grid_search.best_estimator_
val_score = best_model.score(val_data[keep_cols].fillna(0), y_val)
print(f"Validation: {val_score:.1%}")

Running grid search (this will take ~20-30 minutes)...
Fitting 3 folds for each of 648 candidates, totalling 1944 fits

Best params: {'colsample_bytree': 0.7, 'gamma': 1.0, 'learning_rate': 0.05, 'max_depth': 5, 'min_child_weight': 8, 'n_estimators': 100, 'subsample': 0.8}
Best CV score: 60.3%
Validation: 58.1%


In [50]:
# Pre-fetch fight results for KO rate calculation
all_results = pd.read_sql("""
    SELECT 
        ff.fighter_id,
        ff.result,
        f.fight_id,
        f.event_date,
        f.method
    FROM fight_fighters_v2 ff
    JOIN fights_v2 f ON ff.fight_id = f.fight_id
    ORDER BY f.event_date
""", conn)

fighter_results_dict = {
    fid: group.sort_values('event_date').reset_index(drop=True)
    for fid, group in all_results.groupby('fighter_id')
}

print(f"Fighter results cached: {len(fighter_results_dict)}")

Fighter results cached: 4055


In [51]:
# ============================================================
# CELL 16: KO Rate, Body Defense, Ground Defense
# ============================================================

def calculate_ko_rate(fighter_id, as_of_date, window=10):
    """Calculate KO/TKO finish rate in recent fights"""
    
    if fighter_id not in fighter_results_dict:
        return 0
    
    history = fighter_results_dict[fighter_id]
    recent = history[history['event_date'] < as_of_date].tail(window)
    
    if len(recent) == 0:
        return 0
    
    # Time decay weights
    weights = time_decay_weights(recent['event_date'].tolist(), as_of_date, LAM)
    
    # KO/TKO wins (1 if KO/TKO win, 0 otherwise)
    ko_indicators = ((recent['result'] == 'win') & 
                     (recent['method'].str.contains('KO|TKO', case=False, na=False))).astype(int).values
    
    # Weighted average
    ko_rate = np.average(ko_indicators, weights=weights)
    return ko_rate


def calculate_body_defense(fighter_id, as_of_date, window=5):
    """Calculate body strikes absorbed per minute"""
    
    if fighter_id not in fighter_fights_dict:
        return 0
    
    fights = fighter_fights_dict[fighter_id]
    recent = fights[fights['event_date'] < as_of_date].tail(window)
    
    if len(recent) == 0:
        return 0
    
    weights = time_decay_weights(recent['event_date'].tolist(), as_of_date, LAM)
    body_absorbed_list = []
    
    for _, fight in recent.iterrows():
        opp_id = opponents_dict.get((fight['fight_id'], fighter_id))
        if opp_id and opp_id in fighter_fights_dict:
            opp_fight = fighter_fights_dict[opp_id]
            opp_this_fight = opp_fight[opp_fight['fight_id'] == fight['fight_id']]
            if len(opp_this_fight) > 0:
                body_absorbed_list.append(opp_this_fight['body_slpm'].iloc[0])
    
    if len(body_absorbed_list) == 0:
        return 0
    
    return np.average(body_absorbed_list, weights=weights[:len(body_absorbed_list)])


def calculate_ground_defense_adjperf(fighter_id, as_of_date, weight_class, window=5):
    """Calculate ground defense with AdjPerf (opponent-adjusted)"""
    
    if fighter_id not in fighter_fights_dict:
        return 0
    
    fights = fighter_fights_dict[fighter_id]
    recent = fights[fights['event_date'] < as_of_date].tail(window)
    
    if len(recent) == 0:
        return 0
    
    # Get weight class prior for ground_slpm
    wc = 'Welterweight'
    for wc_name in weight_classes:
        if wc_name in str(weight_class):
            wc = wc_name
            break
    
    wc_prior = wc_priors.get(wc, {}).get('ground_slpm')
    if wc_prior is None:
        return 0
    
    z_scores = []
    fight_dates = []
    
    for _, fight in recent.iterrows():
        fight_id = fight['fight_id']
        fight_date = fight['event_date']
        
        # Get opponent
        opp_id = opponents_dict.get((fight_id, fighter_id))
        if opp_id is None:
            continue
        
        # Opponent's ground strikes against this fighter
        if opp_id not in fighter_fights_dict:
            continue
        
        opp_fight = fighter_fights_dict[opp_id]
        opp_this_fight = opp_fight[opp_fight['fight_id'] == fight_id]
        if len(opp_this_fight) == 0:
            continue
        
        observed = opp_this_fight['ground_slpm'].iloc[0]  # Strikes absorbed
        
        # Get opponent's baseline (what they typically land)
        opp_baseline = calculate_opponent_adjperf_baseline(opp_id, 'ground_slpm', fight_date, window=10)
        
        # Calculate z-score
        z = calculate_adjperf_zscore(observed, opp_baseline, wc_prior, shrinkage_k=5)
        z_scores.append(z)
        fight_dates.append(fight_date)
    
    if len(z_scores) == 0:
        return 0
    
    # Time-decay average
    weights = time_decay_weights(fight_dates, as_of_date, LAM)
    return np.average(z_scores, weights=weights)


def add_defensive_features(df):
    rows = []
    
    for idx, fight in df.iterrows():
        if idx % 1000 == 0:
            print(f"  Processing {idx}/{len(df)}...")
        
        row = {'fight_id': fight['fight_id']}
        
        # KO rate
        row['f1_ko_rate'] = calculate_ko_rate(fight['fighter_1_id'], fight['event_date'])
        row['f2_ko_rate'] = calculate_ko_rate(fight['fighter_2_id'], fight['event_date'])
        row['diff_ko_rate'] = row['f1_ko_rate'] - row['f2_ko_rate']
        
        # Body defense
        row['f1_body_absorbed'] = calculate_body_defense(fight['fighter_1_id'], fight['event_date'])
        row['f2_body_absorbed'] = calculate_body_defense(fight['fighter_2_id'], fight['event_date'])
        row['diff_body_absorbed'] = row['f1_body_absorbed'] - row['f2_body_absorbed']
        
        # Ground defense AdjPerf
        row['f1_ground_def_adjperf'] = calculate_ground_defense_adjperf(
            fight['fighter_1_id'], fight['event_date'], fight['weight_class']
        )
        row['f2_ground_def_adjperf'] = calculate_ground_defense_adjperf(
            fight['fighter_2_id'], fight['event_date'], fight['weight_class']
        )
        row['diff_ground_def_adjperf'] = row['f1_ground_def_adjperf'] - row['f2_ground_def_adjperf']
        
        rows.append(row)
    
    defense_df = pd.DataFrame(rows)
    return df.merge(defense_df, on='fight_id', how='left')


print("Adding KO rate, body defense, ground defense features...")
start = time.time()
fight_features_defense = add_defensive_features(fight_features_rounds)
print(f"Done in {time.time()-start:.1f}s")
print(f"Shape: {fight_features_defense.shape}")

Adding KO rate, body defense, ground defense features...
  Processing 0/7800...
  Processing 1000/7800...
  Processing 2000/7800...
  Processing 3000/7800...
  Processing 4000/7800...
  Processing 5000/7800...
  Processing 6000/7800...
  Processing 7000/7800...
Done in 73.7s
Shape: (7800, 178)


In [52]:
keep_cols = [c for c in fight_features_defense.columns 
             if c.startswith('diff_') 
             or c in ['age_diff', 'reach_diff', 'height_diff',
                      'age_ratio', 'reach_ratio', 'height_ratio', 'stance_matchup']]

train_data = fight_features_defense[fight_features_defense['event_date'] < '2021-01-01']
val_data   = fight_features_defense[fight_features_defense['event_date'] >= '2021-01-01']

X_train = train_data[keep_cols].fillna(0)
y_train = train_data['winner']
X_val   = val_data[keep_cols].fillna(0)
y_val   = val_data['winner']

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Features: {len(keep_cols)}")

model = XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    min_child_weight=8, subsample=0.7, colsample_bytree=0.7,
    gamma=1.0, random_state=42
)

model.fit(X_train, y_train)
print(f"\nPrevious: 59.5%")
print(f"Training:   {model.score(X_train, y_train):.1%}")
print(f"Validation: {model.score(X_val, y_val):.1%}")

# Check new feature importance
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

new_features = importance_df[importance_df['feature'].str.contains('ko_rate|body_absorbed|ground_def')]
print("\nNew defensive features ranking:")
print(new_features.to_string())

Train: 6654 | Val: 1146 | Features: 58

Previous: 59.5%
Training:   73.2%
Validation: 58.9%

New defensive features ranking:
                    feature  importance
57  diff_ground_def_adjperf    0.021068
55             diff_ko_rate    0.019657
56       diff_body_absorbed    0.016791


In [53]:
# Nuclear option: Only top 10 features
top_10 = importance_df.head(10)['feature'].tolist()
print(f"Top 10 features: {top_10}")

X_train_minimal = X_train[top_10]
X_val_minimal = X_val[top_10]

model_minimal = XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    min_child_weight=8, subsample=0.7, colsample_bytree=0.7,
    gamma=1.0, random_state=42
)

model_minimal.fit(X_train_minimal, y_train)
print(f"\nWith only top 10 features:")
print(f"Training:   {model_minimal.score(X_train_minimal, y_train):.1%}")
print(f"Validation: {model_minimal.score(X_val_minimal, y_val):.1%}")

Top 10 features: ['age_ratio', 'diff_str_acc', 'age_diff', 'diff_head_acc', 'diff_td_avg', 'diff_total_fights', 'diff_r1_slpm', 'diff_ground_slpm', 'diff_ground_def_adjperf', 'diff_ground_acc']

With only top 10 features:
Training:   67.0%
Validation: 57.9%


In [54]:
# Modern era: 2016 to present (2026)
modern_fights = base_fights[base_fights['event_date'] >= '2016-01-01']
print(f"2016-2026 fights: {len(modern_fights)}")

# Training split (2016-2020)
train_modern = modern_fights[modern_fights['event_date'] < '2021-01-01']
val_modern = modern_fights[modern_fights['event_date'] >= '2021-01-01']

print(f"Train (2016-2020): {len(train_modern)}")
print(f"Val (2021-2026): {len(val_modern)}")

print(f"\nCompare to current:")
print(f"Current train (2007-2020): {len(base_fights[base_fights['event_date'] < '2021-01-01'])}")

2016-2026 fights: 3734
Train (2016-2020): 2588
Val (2021-2026): 1146

Compare to current:
Current train (2007-2020): 6654


In [55]:
# Retrain on modern data only
modern_features = fight_features_rounds[fight_features_rounds['event_date'] >= '2016-01-01']

train_modern = modern_features[modern_features['event_date'] < '2021-01-01']
val_modern = modern_features[modern_features['event_date'] >= '2021-01-01']

keep_cols = [c for c in modern_features.columns 
             if c.startswith('diff_') 
             or c in ['age_diff', 'reach_diff', 'height_diff',
                      'age_ratio', 'reach_ratio', 'height_ratio', 'stance_matchup']]

X_train_modern = train_modern[keep_cols].fillna(0)
X_val_modern = val_modern[keep_cols].fillna(0)

model = XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    min_child_weight=8, subsample=0.7, colsample_bytree=0.7,
    gamma=1.0, random_state=42
)

model.fit(X_train_modern, train_modern['winner'])
print(f"2007-2020 training: 59.5% val")
print(f"\n2016-2020 training (modern only):")
print(f"Training:   {model.score(X_train_modern, train_modern['winner']):.1%}")
print(f"Validation: {model.score(X_val_modern, val_modern['winner']):.1%}")

2007-2020 training: 59.5% val

2016-2020 training (modern only):
Training:   81.4%
Validation: 58.8%
